# 📊 Histórico de Fundos — v4
**Execute célula por célula** (Shift+Enter em cada uma).
Não use 'Executar tudo' — assim você vê o resultado de cada etapa.

In [ ]:
# ── CÉLULA 1: Instalar ───────────────────────────────────────────────
!pip install yfinance openpyxl --quiet
import io, time, requests
import pandas as pd
import yfinance as yf
from datetime import date
from dateutil.relativedelta import relativedelta
print('✔ Pronto')

In [ ]:
# ── CÉLULA 2: Buscar CNPJs pelo nome no cadastro da CVM ──────────────
# Isso resolve de vez o problema de CNPJ errado.

print('Baixando cadastro de fundos da CVM...')
r = requests.get(
    'https://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv',
    timeout=120
)
print(f'Status: {r.status_code} | Tamanho: {len(r.content)/1024/1024:.1f} MB')

cad = pd.read_csv(
    io.StringIO(r.content.decode('latin-1')),
    sep=';',
    dtype={'CNPJ_FUNDO': str},
    usecols=['CNPJ_FUNDO', 'DENOM_SOCIAL', 'SIT']
)

# Filtra só fundos em funcionamento
ativos = cad[cad['SIT'] == 'EM FUNCIONAMENTO NORMAL'].copy()
print(f'Fundos em funcionamento: {len(ativos):,}')

# ── Termos de busca por fundo ────────────────────────────────────────
# Formato: 'Nome exibição': 'termo de busca no cadastro CVM'
BUSCA = {
    # ABRIGO
    'Kinea Inflação Curta':        'KINEA INFLACAO CURTA',
    'Icatu Vanguarda Infl Curta':  'ICATU VANGUARDA INFLACAO CURTA',
    'Kinea Debêntures Incentiv':   'KINEA DEBENTURES INCENTIVADAS',
    'Itaú Debêntures CDI':         'ITAU DEBENTURES INCENTIVADAS CDI',
    'ARX Hedge Infra RF':          'ARX HEDGE FIC INCENTIVADO INFRA',
    'JGP Corporate RF':            'JGP CORPORATE',
    'SPX Seahawk CP':              'SPX SEAHAWK',
    'Kinea Oportunidade FIM':      'KINEA OPORTUNIDADE',
    # RITMO
    'Kinea Atlas II FIM':          'KINEA ATLAS II',
    'MS Global Fixed Income':      'MS GLOBAL FIXED INCOME',
    'Ibiuna Hedge STH':            'IBIUNA HEDGE STH',
    'Kapitalo Kappa':              'KAPITALO KAPPA',
    'Dynamo Cougar':               'DYNAMO COUGAR',
    'Bogari Value':                'BOGARI VALUE',
    'Atmos Ações':                 'ATMOS ACOES',
    'JP Morgan Global Equity':     'JP MORGAN GLOBAL EQUITY',
    'Trend Ouro':                  'TREND OURO',
    'Hashdex Nasdaq Crypto':       'HASHDEX NASDAQ CRYPTO',
    # VANGUARDA
    'Capitânia Premium 45':        'CAPITANIA PREMIUM 45',
    'Valora Guardian':             'VALORA GUARDIAN',
    'Trend Pré-Fixado':            'TREND PRE FIXADO',
    'Kinea Atlas FIM':             'KINEA ATLAS FIM',
    'SPX Nimitz':                  'SPX NIMITZ',
    'JGP Ecossistema 360':         'JGP ECOSSISTEMA',
    'Dahlia Total Return':         'DAHLIA TOTAL RETURN',
    'Encore Long Bias':            'ENCORE LONG BIAS',
}

# Busca cada fundo
CNPJS_ENCONTRADOS = {}
NAO_ENCONTRADOS = []

print('\nBuscando CNPJs por nome:')
for nome, termo in BUSCA.items():
    res = ativos[ativos['DENOM_SOCIAL'].str.upper().str.contains(termo, na=False)]
    if res.empty:
        # tenta sem filtro de situação
        res = cad[cad['DENOM_SOCIAL'].str.upper().str.contains(termo, na=False)]
    if not res.empty:
        cnpj = res.iloc[0]['CNPJ_FUNDO']
        # remove formatação
        cnpj_limpo = ''.join(c for c in cnpj if c.isdigit())
        CNPJS_ENCONTRADOS[nome] = cnpj_limpo
        print(f'  ✔ {nome}: {cnpj} | {res.iloc[0]["DENOM_SOCIAL"]}')
    else:
        NAO_ENCONTRADOS.append(nome)
        print(f'  ❌ {nome}: não encontrado')

print(f'\n→ {len(CNPJS_ENCONTRADOS)}/{len(BUSCA)} fundos com CNPJ confirmado')

In [ ]:
# ── CÉLULA 3: Baixar um mês recente e validar CNPJs ──────────────────
# Confirma que os CNPJs encontrados realmente aparecem nos dados de cotas.

hoje = date.today()
mes_t = hoje.month - 1 if hoje.month > 1 else 12
ano_t = hoje.year if hoje.month > 1 else hoje.year - 1

url = f'https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ano_t}{mes_t:02d}.csv'
print(f'Validando contra: inf_diario_fi_{ano_t}{mes_t:02d}.csv')

r = requests.get(url, timeout=60)
df_val = pd.read_csv(
    io.StringIO(r.content.decode('latin-1')), sep=';',
    usecols=['CNPJ_FUNDO', 'VL_QUOTA'],
    dtype={'CNPJ_FUNDO': str}
)
df_val['CNPJ_FUNDO'] = df_val['CNPJ_FUNDO'].str.replace(r'\D', '', regex=True)
cnpjs_no_arquivo = set(df_val['CNPJ_FUNDO'].values)

print(f'Total de fundos no arquivo de cotas: {len(cnpjs_no_arquivo):,}\n')
print('Validação:')
CNPJS_VALIDOS = {}
for nome, cnpj in CNPJS_ENCONTRADOS.items():
    ok = cnpj in cnpjs_no_arquivo
    status = '✔ OK' if ok else '⚠️  sem cota (fundo master/offshore?)'
    print(f'  {status} — {nome}')
    if ok:
        CNPJS_VALIDOS[nome] = cnpj

print(f'\n→ {len(CNPJS_VALIDOS)} fundos prontos para download')

In [ ]:
# ── CÉLULA 4: Baixar histórico (10 anos, mês a mês) ──────────────────
# Só roda se a Célula 3 mostrou fundos ✔ OK.

DATA_FIM    = date.today().replace(day=1)
DATA_INICIO = (DATA_FIM - relativedelta(years=10)).replace(day=1)

cur, meses = DATA_INICIO, []
while cur <= DATA_FIM:
    meses.append((cur.year, cur.month))
    cur += relativedelta(months=1)

cnpj_map = {v: k for k, v in CNPJS_VALIDOS.items()}
registros = []

print(f'Baixando {len(meses)} meses para {len(cnpj_map)} fundos...')
print('⏱️  Estimativa: 15-20 min\n')

for i, (ano, mes) in enumerate(meses):
    url = f'https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ano}{mes:02d}.csv'
    try:
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            continue
        df = pd.read_csv(
            io.StringIO(r.content.decode('latin-1')), sep=';',
            usecols=['CNPJ_FUNDO', 'DT_COMPTC', 'VL_QUOTA'],
            dtype={'CNPJ_FUNDO': str}
        )
        df['CNPJ_FUNDO'] = df['CNPJ_FUNDO'].str.replace(r'\D', '', regex=True)
        df['DT_COMPTC']  = pd.to_datetime(df['DT_COMPTC'])
        filtro = df[df['CNPJ_FUNDO'].isin(cnpj_map)]
        ultimo = filtro.sort_values('DT_COMPTC').groupby('CNPJ_FUNDO').last().reset_index()
        for _, row in ultimo.iterrows():
            registros.append({
                'nome': cnpj_map[row['CNPJ_FUNDO']],
                'data': row['DT_COMPTC'],
                'cota': float(row['VL_QUOTA'])
            })
    except Exception as e:
        pass
    if i % 12 == 0:
        print(f'  {mes:02d}/{ano} — {len(registros)} registros acumulados')
    time.sleep(0.2)

print(f'\n✔ Total: {len(registros)} registros coletados')

In [ ]:
# ── CÉLULA 5: ETFs + Benchmarks ──────────────────────────────────────

ETFS = {
    'IVVB11': 'IVVB11.SA',
    'SMAL11': 'SMAL11.SA',
    'IMAB11': 'IMAB11.SA',
    'IB5M11': 'IB5M11.SA',
    'GOLD11': 'GOLD11.SA',
    'HASH11': 'HASH11.SA',
}

DATA_INI_STR = DATA_INICIO.strftime('%Y-%m-%d')
DATA_FIM_STR = date.today().strftime('%Y-%m-%d')

print('ETFs:')
for nome, ticker in ETFS.items():
    try:
        df_e = yf.download(ticker, start=DATA_INI_STR, end=DATA_FIM_STR,
                           interval='1mo', progress=False, auto_adjust=True)
        if df_e.empty: print(f'  ❌ {nome}'); continue
        precos = df_e['Close'].squeeze()
        for dt, val in precos.items():
            registros.append({'nome': nome, 'data': pd.Timestamp(dt), 'cota': float(val)})
        print(f'  ✔ {nome}: {len(precos)} meses')
    except Exception as e:
        print(f'  ❌ {nome}: {e}')

print('\nBenchmarks:')
def bcb(cod, nome):
    try:
        url = (f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{cod}/dados'
               f'?formato=json&dataInicial={DATA_INICIO.strftime("%d/%m/%Y")}'
               f'&dataFinal={date.today().strftime("%d/%m/%Y")}')
        df = pd.DataFrame(requests.get(url, timeout=15).json())
        df['data']  = pd.to_datetime(df['data'], format='%d/%m/%Y')
        df['valor'] = df['valor'].astype(float)
        # Para CDI/IPCA já vem como % mensal — armazena direto como "cota" fictícia
        for _, row in df.iterrows():
            registros.append({'nome': nome, 'data': row['data'], 'cota': row['valor']})
        print(f'  ✔ {nome}: {len(df)} meses')
    except Exception as e:
        print(f'  ❌ {nome}: {e}')

bcb(4391, 'CDI')
bcb(433,  'IPCA')

In [ ]:
# ── CÉLULA 6: Calcular rendimento e gerar Excel ───────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BENCHMARKS = {'CDI', 'IPCA'}
ETFS_SET   = set(ETFS.keys())

df_raw = pd.DataFrame(registros).sort_values(['nome', 'data'])

linhas = []
for nome, grp in df_raw.groupby('nome'):
    serie = grp.set_index('data')['cota'].sort_index()

    if nome in BENCHMARKS:
        # CDI/IPCA já vêm como % mensal
        ret = serie
        ret.index = ret.index.to_period('M').to_timestamp('M')
    elif nome in ETFS_SET:
        # ETFs: calcular retorno mensal a partir do preço
        ret = serie.pct_change().dropna() * 100
        ret.index = ret.index.to_period('M').to_timestamp('M')
    else:
        # Fundos CVM: cota → retorno mensal
        if len(serie) < 2: continue
        ret = serie.pct_change().dropna() * 100
        ret.index = ret.index.to_period('M').to_timestamp('M')

    ret = ret[~ret.index.duplicated(keep='last')]
    for dt, val in ret.items():
        linhas.append({
            'Data':               dt.strftime('%m/%Y'),
            'Ativo':              nome,
            'Rendimento_Mensal_%': round(float(val), 4)
        })

df_final = pd.DataFrame(linhas).drop_duplicates(['Data','Ativo'])
df_final = df_final.sort_values(['Ativo','Data']).reset_index(drop=True)

print(f'Linhas totais: {len(df_final):,}')
print(f'Ativos: {df_final["Ativo"].nunique()}')
print(df_final.groupby("Ativo")["Data"].count().to_string())

# ── Excel ─────────────────────────────────────────────────────────────
ARQUIVO = 'historico_fundos.xlsx'
with pd.ExcelWriter(ARQUIVO, engine='openpyxl') as writer:
    # Tabela longa principal
    df_final.to_excel(writer, sheet_name='Historico', index=False)
    # Pivotada (datas nas linhas, ativos nas colunas)
    piv = df_final.pivot(index='Data', columns='Ativo', values='Rendimento_Mensal_%')
    piv.to_excel(writer, sheet_name='Pivotado')

# Formatar
wb = load_workbook(ARQUIVO)
for ws in wb.worksheets:
    fill = PatternFill('solid', start_color='D6E4F0')
    for cell in ws[1]:
        if cell.value:
            cell.font = Font(bold=True, name='Arial')
            cell.fill = fill
            cell.alignment = Alignment(horizontal='center')
    for col in ws.columns:
        w = max((len(str(c.value or '')) for c in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w+2, 28)
    ws.freeze_panes = 'B2'
wb.save(ARQUIVO)
print(f'\n✅ {ARQUIVO} gerado!')

In [ ]:
# ── CÉLULA 7: Download ────────────────────────────────────────────────
from google.colab import files
files.download('historico_fundos.xlsx')
print('📥 Download iniciado!')